In [ ]:
import os
API_TOKEN = os.getenv("API_TOKEN")

import re
import json
import math
import unicodedata
from datetime import datetime
from typing import TypedDict, List, Dict, Any

import torch
import pandas as pd

from langgraph.graph import StateGraph, END
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from sentence_transformers import SentenceTransformer

from docx import Document
from striprtf.striprtf import rtf_to_text

MODE = 'keyword'   # 'keyword' or 'title'

THESAURUS_XLSX = './Thesaurus/SK_Local_Thesaurus.xlsx'
THESAURUS_COL  = 'slovak_terms'
INPUT_DIR      = './Input'
OUTPUT_DIR     = './Output'
PERSIST_DIR    = './chroma_store'   # persistent Chroma root

EMBED_MODEL    = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
LLM_MODEL      = 'nvidia/Llama-3.1-Nemotron-Nano-8B-v1'
#LLM_MODEL      = 'nvidia/NVIDIA-Nemotron-Nano-9B-v2'

CHUNK_TOKENS   = 320
CHUNK_OVERLAP  = 48
THINKING       = 'on'

# Agent gates
CONF_GATE      = 0.70    # confidence needed to stop
MAX_RETRIES    = 3

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Stopwords

def load_stopwords():
    path = './Input/stopword_dictionary.txt'
    try:
        with open(path, 'r', encoding='utf-8') as f:
            words = {line.strip().lower() for line in f if line.strip()}
        print(f'Loaded {len(words)} stopwords from {path}')
        return words
    except FileNotFoundError:
        print(f'No stopword file at {path}. Continuing without.')
        return set()
    except Exception as e:
        print(f'Failed to load stopwords: {e}')
        return set()

STOPWORDS = load_stopwords()

# Thesaurus (Excel) → canonical terms + robust matchers
def strip_accents(s: str) -> str:
    return ''.join(ch for ch in unicodedata.normalize('NFD', s) if not unicodedata.combining(ch))

def _canon_left_of_dash(term: str) -> str:
    # normalize "X - pozri Y" → keep left
    return term.split(' - ')[0].strip()

def _split_parenthetical_synonyms(term: str):
    # "Spôsobilosť ... (svojprávnosť)" → base + ["svojprávnosť"]
    m = re.match(r'^(.*?)(\s*\(([^)]+)\))\s*$', term)
    if not m:
        return term.strip(), []
    base = m.group(1).strip()
    inside = m.group(3).strip()
    syns = [s.strip() for s in re.split(r'[\/,;]', inside) if s.strip()]
    return base, syns

def _wildify(s: str) -> str:
    # robust regex: spaces → \s+, dashes → [-–—]
    s = s.strip()
    s = re.sub(r'\s+', r'\\s+', re.escape(s))
    return s.replace(r'\-', r'[-–—]')

def build_term_matcher(term: str):
    canon = _canon_left_of_dash(term)
    base, syns = _split_parenthetical_synonyms(canon)

    # patterns with diacritics
    pats = [re.compile(r'\b' + _wildify(base) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    for syn in syns:
        pats.append(re.compile(r'\b' + _wildify(syn) + r'\b', re.IGNORECASE))

    # patterns without diacritics, matched against accent-stripped text
    pats_na = [re.compile(r'\b' + _wildify(strip_accents(base)) + r'(?:\s*\([^)]*\))?\b', re.IGNORECASE)]
    pats_na += [re.compile(r'\b' + _wildify(strip_accents(syn)) + r'\b', re.IGNORECASE) for syn in syns]

    return {'canon': canon, 'patterns': pats, 'patterns_noacc': pats_na, 'synonyms': syns}

def load_thesaurus_terms(xlsx_path=THESAURUS_XLSX, column=THESAURUS_COL):
    if not os.path.exists(xlsx_path):
        raise FileNotFoundError(f'Thesaurus file not found: {xlsx_path}')
    df = pd.read_excel(xlsx_path, engine='openpyxl')
    if column not in df.columns:
        raise ValueError(f'Column "{column}" not found. Columns: {list(df.columns)}')

    raw = df[column].dropna().astype(str).tolist()
    parts = []
    for cell in raw:
        parts.extend(re.split(r'[,\n;]+', cell))

    seen = set()
    terms = []
    for t in parts:
        t = t.strip()
        if not t:
            continue
        low = t.lower()
        if len(low) == 1 or low in STOPWORDS:
            continue
        if t not in seen:
            seen.add(t)
            terms.append(t)
    print(f'Thesaurus terms loaded: {len(terms)}')
    return terms

THESAURUS_TERMS = load_thesaurus_terms()
TERM_MATCHERS   = { (m := build_term_matcher(t))['canon'].lower(): m for t in THESAURUS_TERMS }
CANON_TERMS     = list(TERM_MATCHERS.keys())

In [ ]:
# Vector stores (thesaurus + session corpus) with modern packages
# Run all vector-store embeddings on CPU to save VRAM for the LLM
emb_lc = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"}
)

thesaurus_store = Chroma(
    collection_name='thesaurus',
    persist_directory=PERSIST_DIR,
    embedding_function=emb_lc,
)
corpus_store = Chroma(
    collection_name='session_corpus',
    persist_directory=PERSIST_DIR,
    embedding_function=emb_lc,
)

# Thesaurus: add only missing IDs (add_texts isn't upsert)
existing_ids = set(thesaurus_store.get().get('ids', []))
to_texts, to_ids = [], []
for t in THESAURUS_TERMS:
    tid = f"term::{t}"
    if tid not in existing_ids:
        to_texts.append(t)
        to_ids.append(tid)
if to_texts:
    thesaurus_store.add_texts(texts=to_texts, ids=to_ids, metadatas=[{'type':'term'}]*len(to_texts))

# Clear session corpus for this run
cur_ids = corpus_store.get().get('ids', [])
if cur_ids:
    corpus_store.delete(ids=cur_ids)

thesaurus_ret = thesaurus_store.as_retriever(search_kwargs={'k': 10})
corpus_ret    = corpus_store.as_retriever(search_kwargs={'k': 8})


# LLM (Nemotron) with memory-aware quantization
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU required.')

total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
force_4bit   = os.environ.get('FORCE_4BIT', 'false').lower() == 'true'

if total_mem_gb < 6:
    qconf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                                bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4')
elif total_mem_gb < 16 or force_4bit:
    qconf = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                                bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4')
else:
    qconf = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL, device_map='auto', quantization_config=qconf,
    torch_dtype=torch.bfloat16, trust_remote_code=True
)
tok = AutoTokenizer.from_pretrained(LLM_MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = 'left'

gen = pipeline(
    'text-generation',
    model=model,
    tokenizer=tok,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    max_new_tokens=128,       # 128 keeps VRAM comfy; raise if needed
    do_sample=True,
    temperature=0.1,
    top_p=0.7,
    repetition_penalty=1.3,
)
llm = HuggingFacePipeline(pipeline=gen)

# Global SentenceTransformer on CPU for verification scoring
EMBED_ST = SentenceTransformer(EMBED_MODEL, device="cpu")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
# Document loading & sectioning

_HEADING_NAME_PREFIXES = ('heading', 'nadpis','po-heading-id')
_HEADING_EXACT_NAMES   = {'title','subtitle','toc heading','obsah','po-heading-id_'}
_PAGE_LINE_RE = re.compile(r'^\s*(strana|page)\s+\d+(?:\s*/\s*\d+)?\s*$', re.IGNORECASE)
_RUBRIC_RE    = re.compile(r'([.\-–—]{3,})')

def _gather_header_footer_text(doc: Document):
    lines = []
    for sec in doc.sections:
        for part in (sec.header, sec.footer):
            for p in part.paragraphs:
                t = p.text.strip()
                if t: lines.append(t)
            for tb in part.tables:
                for row in tb.rows:
                    for cell in row.cells:
                        for p in cell.paragraphs:
                            t = p.text.strip()
                            if t: lines.append(t)
    return set(lines)

def _looks_like_heading_style(p):
    try: name = (p.style.name or '').strip().lower()
    except: name = ''
    try: sid  = (getattr(p.style,'style_id','') or '').lower()
    except: sid = ''
    if any(name.startswith(pref) for pref in _HEADING_NAME_PREFIXES): return True
    if name in _HEADING_EXACT_NAMES: return True
    if re.match(r'^heading\d+$', name) or re.match(r'^nadpis\s*\d+$', name): return True
    if re.match(r'^heading\d+$', sid): return True
    return False

def _is_header_footer_like(line: str):
    l = line.strip()
    if not l: return False
    if _PAGE_LINE_RE.match(l): return True
    if _RUBRIC_RE.search(l): return True
    if len(l.split()) <= 6 and l == l.upper() and any(ch.isalpha() for ch in l): return True
    return False

def split_docx_by_headers(path):
    try:
        doc = Document(path)
        hf  = _gather_header_footer_text(doc)
        sections, current, buf, full = [], None, [], []
        for p in doc.paragraphs:
            t = p.text.strip()
            if not t: continue
            if t in hf or _is_header_footer_like(t): continue
            if _looks_like_heading_style(p):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = t, []
            else:
                full.append(t)
                if current is not None:
                    buf.append(t)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'DOCX split failed {path}: {e}')
        return []

def _is_heading_line(l: str):
    words = l.split()
    if len(words) <= 8:
        alpha = sum(ch.isalpha() for ch in l)
        if alpha and sum(ch.isupper() for ch in l if ch.isalpha())/alpha >= 0.7: return True
        if re.match(r'^\d+(\.\d+)*\s+\S', l): return True
    return False

def split_rtf_by_headers(path):
    try:
        raw = rtf_to_text(open(path,'r',encoding='utf-8').read())
        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        lines = [ln for ln in lines if not _is_header_footer_like(ln)]
        sections, current, buf, full = [], None, [], []
        for ln in lines:
            if _is_heading_line(ln):
                if current is not None and buf:
                    sections.append((current, '\n'.join(buf).strip()))
                current, buf = ln, []
            else:
                full.append(ln)
                if current is not None:
                    buf.append(ln)
        if current is not None and buf:
            sections.append((current, '\n'.join(buf).strip()))
        return [('__DOCUMENT__', '\n'.join(full).strip())] + sections
    except Exception as e:
        print(f'RTF split failed {path}: {e}')
        return []

#def simple_tokenize(s: str):
#    return re.findall(r"[A-Za-zÁÄČĎÉÍĽŇÓÔŔŠŤÚÝŽáäčďéíľňóôŕšťúýž0-9\-]+", s)

#def chunk_text(s: str, size=CHUNK_TOKENS, overlap=CHUNK_OVERLAP):
#    toks = simple_tokenize(s)
#    if not toks:
#        return []
#    step = max(1, size - overlap)
#    return [' '.join(toks[i:i+size]).strip() for i in range(0, len(toks), step)
#            if ' '.join(toks[i:i+size]).strip()]

def index_session_corpus(input_dir=INPUT_DIR):
    files = sorted(os.listdir(input_dir), key=str.lower)
    total = 0
    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith('.docx'):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith('.rtf'):
            secs = split_rtf_by_headers(p)
        else:
            continue
        for header, text in secs:
            if not text.strip(): continue
            #chunks = chunk_text(text)
            #if not chunks: continue
            #ids = [uuid.uuid4().hex for _ in chunks]
            #metas = [{'file': f, 'header': header, 'kind': 'chunk'} for _ in chunks]
            #corpus_store.add_texts(texts=chunks, ids=ids, metadatas=metas)
            #total += len(chunks)
    print(f'Indexed {total} chunks to session_corpus.')

index_session_corpus(INPUT_DIR)

# Occurrence counting & candidate generation (keyword mode)

def count_occurrences(text: str, matcher) -> int:
    if not text:
        return 0
    a = sum(len(p.findall(text)) for p in matcher['patterns'])
    b = sum(len(p.findall(strip_accents(text))) for p in matcher['patterns_noacc'])
    return max(a, b)

def occ_candidates(text: str, topk=5):
    occ = {}
    for canon, m in TERM_MATCHERS.items():
        if len(canon) < 3 or canon in STOPWORDS:
            continue
        c = count_occurrences(text, m)
        if c > 0:
            occ[canon] = c
    return [k for k,_ in sorted(occ.items(), key=lambda kv: (-kv[1], -len(kv[0])))[:topk]]

def llm_candidates(text: str, th_terms: List[str], topk=5):
    seed = '\n'.join(th_terms[:40])
    prompt = (
        "Navrhni TOP právne pojmy (1-4 slová) relevantné k textu. "
        "Použi zoznam pojmov ako inšpiráciu. Vráť iba JSON zoznam reťazcov.\n\n"
        f"POJMY:\n{seed}\n\nTEXT:\n{text}\n\nODPOVEĎ:"
    )
    out = llm.invoke(prompt)
    try:
        js = json.loads(out.strip().split('[')[-1].split(']')[0].join(['[', ']']))
    except Exception:
        js = [w.strip(' "„”') for w in re.split(r'[,;\n]', out) if 0 < len(w.strip()) <= 48]
    cands = []
    for w in js:
        base = _canon_left_of_dash(w).lower().strip()
        if base:
            cands.append(base)
    dedup, seen = [], set()
    for c in cands:
        if c not in seen:
            dedup.append(c); seen.add(c)
    return dedup[:topk]

# TITLE MODE HELPERS
def format_title(s: str) -> str:
    """Clean up spacing and trailing punctuation."""
    s = (s or '').strip()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'[\.，。…]+$', '', s)  # no trailing dots
    return s

def terms_matched_in_text(text: str) -> set:
    """Which canonical thesaurus terms actually occur in the text."""
    hits = set()
    if not text: 
        return hits
    t_na = strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        if max(a, b) > 0:
            hits.add(canon)
    return hits

def _title_len_score(title: str) -> float:
    """Prefer 3-12 words; soft penalty otherwise."""
    n = len(title.split())
    if 3 <= n <= 12: 
        return 1.0
    return max(0.0, 1.0 - abs(n - 7) * 0.1)

def _contains_canon(title: str, canon: str) -> bool:
    """Accent-insensitive substring match of canonical term within title."""
    return strip_accents(canon).lower() in strip_accents(title).lower()

def score_title_candidate(title: str, section_text: str):
    """
    Score a title by:
        - sim_score: cosine similarity(title, section_text) via CPU ST embeddings
        - cov_score: coverage of canonical terms in section
        - len_score: 3-12 words sweet spot
    Final = 0.6*sim + 0.3*cov + 0.1*len
    """
    title = format_title(title)
    if not title or not section_text:
        return 0.0, 0.0, 0.0

    tv = EMBED_ST.encode([title], convert_to_numpy=True, normalize_embeddings=True)[0]
    sv = EMBED_ST.encode([section_text], convert_to_numpy=True, normalize_embeddings=True)[0]
    sim = float((tv @ sv).item())              # [-1, 1]
    sim_score = max(0.0, (sim + 1.0) / 2.0)    # -> [0, 1]

    in_text = terms_matched_in_text(section_text)
    covered = sum(1 for c in in_text if _contains_canon(title, c))
    cov_score = (covered / max(1, len(in_text))) if in_text else 0.0

    len_score = _title_len_score(title)
    final = 0.6 * sim_score + 0.3 * cov_score + 0.1 * len_score
    return final, cov_score, sim

def title_candidates_from_llm(text: str, th_terms: list[str], topk: int = 6) -> list[str]:
    seed = '\n'.join(th_terms[:60])
    prompt = (
        "Navrhni 3-6 návrhov právneho nadpisu (hlavičky) pre nasledujúci text.\n"
        "Požiadavky: presný, zrozumiteľný, 3-12 slov, bez bodky na konci.\n"
        "Uprednostni termíny z tezauru, ale môžeš doplniť opisné slová.\n"
        "Vráť IBA JSON zoznam reťazcov (bez komentárov).\n\n"
        f"TEZAURUS (výber):\n{seed}\n\n"
        f"TEXT:\n{text}\n\n"
        "ODPOVEĎ:"
    )
    out = llm.invoke(prompt)
    try:
        js = json.loads(out.strip().split('[')[-1].split(']')[0].join(['[', ']']))
        cands = [format_title(s) for s in js if isinstance(s, str)]
    except Exception:
        rough = [w.strip(' "„”') for w in re.split(r'[\n;]+', out)]
        cands = [format_title(w) for w in rough if 3 <= len(w.split()) <= 12]
    seen, dedup = set(), []
    for t in cands:
        key = strip_accents(t).lower()
        if t and key not in seen:
            seen.add(key); dedup.append(t)
    return dedup[:topk]

def title_candidates(text: str, topk: int = 6) -> list[str]:
    # 1) From thesaurus hits as "plain" titles
    occ = {}
    t_na = strip_accents(text)
    for canon, m in TERM_MATCHERS.items():
        a = sum(len(p.findall(text)) for p in m['patterns'])
        b = sum(len(p.findall(t_na)) for p in m['patterns_noacc'])
        cnt = max(a, b)
        if cnt > 0:
            occ[canon] = cnt
    seeds = [format_title(k) for k, _ in sorted(occ.items(), key=lambda kv: (-kv[1], -len(kv[0])))[:max(3, topk//2)]]

    # 2) LLM suggestions
    llm_seeds = title_candidates_from_llm(text, CANON_TERMS, topk=topk)

    # Merge + dedupe
    seen, merged = set(), []
    for t in (seeds + llm_seeds):
        key = strip_accents(t).lower()
        if t and key not in seen:
            seen.add(key); merged.append(t)
    return merged[:topk]

# -------------------- Retrieval + Scoring helpers (keyword mode) --------------------
def retrieve_hits(q: str, k: int = 6):
    """
    Retrieve top-k chunks from the session corpus.
    Returns: (docs, citations_shortlist)
    """
    docs = corpus_ret.invoke(q) or []
    docs = docs[:k]
    cites = []
    for d in docs[:2]:
        md = d.metadata or {}
        cites.append({
            "file": md.get("file", "?"),
            "header": md.get("header", "?"),
            "snippet": (d.page_content or "")[:180]
        })
    return docs, cites

def score_candidate(term: str, section_text: str):
    """
    Hybrid score on CPU (keyword mode):
        - occ_score from regex occurrences in the section
        - sim_score from best cosine similarity between term and retrieved chunks
    """
    m = TERM_MATCHERS.get(term)
    occ = count_occurrences(section_text, m) if m else 0

    docs, _ = retrieve_hits(term, k=4)
    if not docs:
        best_sim = -1.0
    else:
        q_vec = EMBED_ST.encode([term], convert_to_numpy=True, normalize_embeddings=True)[0]
        contents = [(d.page_content or "") for d in docs]
        d_mat = EMBED_ST.encode(contents, convert_to_numpy=True, normalize_embeddings=True)
        sims = (d_mat @ q_vec).astype("float32")  # cosine since normalized
        best_sim = float(sims.max()) if sims.size else -1.0

    occ_score = 1 - math.exp(-occ)               # 0..1
    sim_score = max(0.0, (best_sim + 1.0) / 2.0) # [-1,1] -> [0,1]
    final = 0.6 * occ_score + 0.4 * sim_score
    return final, occ, best_sim

# LangGraph: nodes and wiring (with proper retry node)

class AgentState(TypedDict):
    file: str
    header: str
    text: str
    loop: int
    candidates: List[str]
    used_queries: List[str]
    citations: List[Dict[str, Any]]
    scores: Dict[str, Dict[str, float]]   # term/title -> {score, ...}
    final: Dict[str, Any]                 # {top_keyword/title, confidence, method, occurrences}

def node_seed(state: AgentState):
    txt = state["text"]
    if MODE == 'title':
        seeds = title_candidates(txt, topk=6)  # drop to 4 if you want it lighter
    else:
        seeds = occ_candidates(txt, topk=5)
        if len(seeds) < 5:
            seeds += [c for c in llm_candidates(txt, CANON_TERMS, topk=5) if c not in seeds]
        seeds = seeds[:5]
    return {"candidates": seeds, "used_queries": seeds}

def node_retrieve(state: AgentState):
    # Retrieval is only meaningful for keyword scoring; in title mode we skip citations
    cand = state["candidates"]
    citations = list(state.get("citations", []))
    if MODE == 'keyword':
        for t in cand:
            _, cites = retrieve_hits(t, k=6)
            citations.extend(cites)
    return {"citations": citations}

def node_score(state: AgentState):
    txt = state["text"]
    scored = {}
    for t in state["candidates"]:
        if MODE == 'title':
            s, cov, sim = score_title_candidate(t, txt)
            scored[t] = {"score": s, "coverage": cov, "best_sim": sim}
        else:
            s, occ, sim = score_candidate(t, txt)
            scored[t] = {"score": s, "occ": occ, "best_sim": sim}
    return {"scores": scored}

def node_reflect(state: AgentState):
    scored = state["scores"]; txt = state["text"]

    if MODE == 'title':
        table = "\n".join([f"- {t}: score={v['score']:.3f}, cov={v.get('coverage',0):.3f}, sim={v['best_sim']:.3f}"
                            for t, v in scored.items()])
        prompt = (
            "Zvoľ JEDEN najvhodnejší právny nadpis (3–12 slov) zo zoznamu. "
            "Vráť len nadpis bez bodky.\n\n"
            f"Kandidáti:\n{table}\n\nTEXT:\n{txt}\n\nNADPIS:"
        )
        out = llm.invoke(prompt).strip()
        title = format_title(re.sub(r'^[\-\•\:\s]+','', out))
        score = scored.get(title, {}).get("score", 0.0)
        return {"final": {"top_keyword": title, "confidence": float(score),
                            "method": "langgraph-agent-title", "occurrences": 0}}
    else:
        table = "\n".join([f"- {t}: score={v['score']:.3f}, occ={v['occ']}, sim={v['best_sim']:.3f}"
                            for t, v in scored.items()])
        prompt = (
            "Zvoľ JEDEN najvhodnejší právny pojem (1–4 slová) na základe skóre a textu. "
            "Vráť len pojem, bez komentára.\n\n"
            f"Kandidáti:\n{table}\n\nTEXT:\n{txt}\n\nPOJEM:"
        )
        out = llm.invoke(prompt).strip()
        cand = re.sub(r'^[\-\•\:\s]+','', out)
        cand = re.sub(r'[\.，。]+$','', cand).lower().strip()
        score = scored.get(cand, {}).get("score", 0.0)
        occ   = scored.get(cand, {}).get("occ", 0)
        return {"final": {"top_keyword": cand, "confidence": float(score),
                            "method": "langgraph-agent", "occurrences": int(occ)}}

def _expand_candidates(state: AgentState):
    chosen = state["final"].get("top_keyword", "")
    expansions = []
    if MODE == 'keyword' and chosen in TERM_MATCHERS:
        expansions += TERM_MATCHERS[chosen]['synonyms']
    hdr = state.get("header")
    if hdr and hdr != "__DOCUMENT__":
        expansions += [w.strip().lower() for w in re.split(r'[,\-–—:;]', hdr) if len(w.strip()) > 2]
    expansions = [e for e in expansions if e][:4]

    seen = set()
    combined = []
    for c in list(state["candidates"]) + expansions:
        c = c.strip()
        if not c or c.lower() in seen:
            continue
        combined.append(c); seen.add(c.lower())
    return combined[:5], expansions

def node_retry(state: AgentState):
    new_cands, new_queries = _expand_candidates(state)
    return {
        "candidates": new_cands,
        "used_queries": state["used_queries"] + new_queries,
        "loop": state["loop"] + 1,
        "scores": {},                    # optional: clear previous scores
        "citations": state["citations"]  # keep citations history
    }

def gate(state: AgentState):
    conf = state["final"].get("confidence", 0.0)
    if conf >= CONF_GATE or state["loop"] >= MAX_RETRIES:
        return "end"
    return "retry"

# Graph build
graph = StateGraph(AgentState)
graph.add_node("seed", node_seed)
graph.add_node("retrieve", node_retrieve)
graph.add_node("score", node_score)
graph.add_node("reflect", node_reflect)
graph.add_node("retry", node_retry)

graph.set_entry_point("seed")
graph.add_edge("seed", "retrieve")
graph.add_edge("retrieve", "score")
graph.add_edge("score", "reflect")
graph.add_conditional_edges("reflect", gate, {"retry": "retry", "end": END})
graph.add_edge("retry", "retrieve")

app = graph.compile()

# Driver

def agent_run(file: str, header: str, text: str):
    state: AgentState = {
        "file": file, "header": header, "text": text,
        "loop": 0, "candidates": [], "used_queries": [], "citations": [], "scores": {}, "final": {}
    }
    out = app.invoke(state)
    return out["final"], out.get("used_queries", []), out.get("citations", [])

def process_all(input_dir=INPUT_DIR):
    rows = []
    files = sorted(os.listdir(input_dir), key=str.lower)

    # optional hygiene before big loop
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    for f in files:
        p = os.path.join(input_dir, f)
        if f.lower().endswith('.docx'):
            secs = split_docx_by_headers(p)
        elif f.lower().endswith('.rtf'):
            secs = split_rtf_by_headers(p)
        else:
            continue
        if not secs:
            print(f'No text in {f}')
            continue

        for header, text in secs:
            if not text.strip():
                continue
            final, used, _ = agent_run(f, header, text)
            if MODE == 'title':
                rows.append({
                    "file": f,
                    "header": header,
                    "suggested_title": final.get("top_keyword", ""),
                    "confidence": round(final.get("confidence", 0.0), 3),
                    "method": final.get("method", ""),
                    "used_queries": "; ".join(used),
                    "thinking": THINKING
                })
            else:
                rows.append({
                    "file": f,
                    "header": header,
                    "top_keyword": final.get("top_keyword", ""),
                    "confidence": round(final.get("confidence", 0.0), 3),
                    "method": final.get("method", ""),
                    "occurrences": final.get("occurrences", 0),
                    "used_queries": "; ".join(used),
                    "thinking": THINKING
                })
        print(f'Processed {f} with {len(secs)} sections.')

        # free fragmented VRAM between files
        torch.cuda.empty_cache()

    today = datetime.today().strftime("%Y-%m-%d")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    if MODE == 'title':
        df = pd.DataFrame(rows, columns=[
            "file","header","suggested_title","confidence","method","used_queries","thinking"
        ])
        out_stub = "langgraph_agentic_titles"
    else:
        df = pd.DataFrame(rows, columns=[
            "file","header","top_keyword","confidence","method","occurrences","used_queries","thinking"
        ])
        out_stub = "langgraph_agentic_top_keyword"

    #csv_path  = os.path.join(OUTPUT_DIR, f"{out_stub}_{today}.csv")
    xlsx_path = os.path.join(OUTPUT_DIR, f"{out_stub}_{today}.xlsx")
    #df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    try:
        df.to_excel(xlsx_path, index=False, engine="openpyxl")
    except Exception as e:
        print(f"[WARN] Excel write failed: {e}")
    #print(f"Saved: {csv_path}")
    return df

if __name__ == "__main__":
    df = process_all(INPUT_DIR)
    try:
        from IPython.display import display
        display(df.head(20))
    except Exception:
        print(df.to_string(index=False))
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()